In [ ]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"]='2'  # if you want to specify which GPU to use in the notebook

import sys
sys.path.append("..")
from dexter import DEXTER, RunConfig
import torch

In [ ]:
device = "cuda"
cfg = RunConfig(optim_steps=100)

models we can use and sample modified models to compare against: (run just the selected cell)

In [ ]:
# sd1.4
cfg.base_model = "CompVis/stable-diffusion-v1-4"

# modifications:

# unlearning - ac
cfg.modified_model = "/home-local/hector-local/Projects/leonardo_code/weights/localscratch/damon2024/DM_baselines/ALL_baseline_ckpt/nudity/AC/AC-Nudity-Diffusers-UNet-xattn.pt"
cfg.load_type = 'weights'
cfg.modified_model_strict_load = False

# unlearning - spm
# cfg.modified_model = "/home-local/hector-local/Projects/leonardo_code/weights/localscratch/damon2024/DM_baselines/ALL_baseline_ckpt/nudity/SPM/SPM-Nudity-Diffusers-UNet.pt"
# cfg.load_type = 'weights'
# cfg.modified_model_strict_load = True

In [ ]:
# sd1.5
cfg.base_model = "stable-diffusion-v1-5/stable-diffusion-v1-5"

# customization - db
cfg.modified_model = "/home-local/hector-local/Projects/leonardo_code/weights/customization/sd15/customconcept101/dbft-prior/pet_dog4/unet/diffusion_pytorch_model.safetensors"
cfg.load_type = 'weights'
cfg.modified_model_strict_load = True

# customization - cd
# cfg.modified_model = "/home-local/hector-local/Projects/leonardo_code/weights/customization/sd15/customconcept101/cdlora-prior/pet_dog4"
# cfg.load_type = 'lora'

In [ ]:
# sd2.1
cfg.base_model = "sd2-community/stable-diffusion-2-1"
cfg.clip_orig_text_encoder_name = "sd2-community/stable-diffusion-2-1"
cfg.clip_orig_text_encoder_name_subfolder = "text_encoder"
cfg.clip_orig_tokenizer_name_subfolder = "tokenizer"
cfg.load_type = 'lora'
# modifications:

# dblora
cfg.modified_model = "/disk1/users/hector-local/Projects/diffusion-forgetting/output/model/sd21/customconcept101/dblora-prior/pet_dog4"
cfg.load_type = 'lora'

rest of the config:

In [ ]:
cfg.torch_dtype = 'bfloat16'
cfg.variant = None
cfg.outdir = 'out_debug'

cfg.objective_direction = 'maximize'
cfg.mask_type = 'single_mask'
# cfg.mask_type = 'multi_mask'
cfg.num_masks = 6 # when using multi_mask

cfg.bs = 1
cfg.bs = 4
cfg.diff_steps = 50  # the diffusion steps
cfg.guidance_scale = 7.5
cfg.gen_dim = 512
cfg.gen_steps = 4  # the actual generation steps the models do before stopping and computing the loss
# cfg.gen_steps = 25
cfg.seed = 42
# cfg.seed = None
cfg.save_images = False
cfg.convergence_threshold = -4  # grows with gen_steps. very low for no early stopping. use as needed.

# cfg.gradient_checkpointing = True  # saves memory, no speed hit for our use case!

run

In [ ]:
clf, tfms = None, None
dexter = DEXTER(cfg, device=device)

In [ ]:
run_dir = dexter.analyze_classifier(clf, tfms)

# VRAM usage

## SD1.4

mask | steps | bs | vram
---- | ----- | -- | -----
single | 2 | 1 | 20674M
single | 4 | 1 | 24870M
single | 8 | 1 | 29206M
single | 12 | 1 | 36472M
single | 16 | 1 | 43678M
multi | 2 | 1 | 24934M
single | 4 | 2 | 28682M
single | 4 | 4 | 42866M

with gradient checkpointing
unet | t.enc | mask | steps | bs | vram
---- | ----- | ---- | ----- | -- | -----
yes  | no| single | 4 | 4 | 22964M
yes  | yes| single | 4 | 4 | 23130M
yes  | no| single | 8 | 4 | 24968M
yes  | no| single | 16 | 4 | 34368M
yes  | no| single | 25 | 4 | 44990M
yes  | no| single | 32 | 4 | - OOM

## SD2.1

mask | steps | bs | vram
---- | ----- | -- | -----
single | 4 | 1 | 23462M
single | 4 | 4 | 38510M